## Configuration

Use `FEATURE_EXTRACTORS_TO_RUN` to select `tfidf`, `glove`, or both. Use `MODELS_TO_RUN` to select Logistic Regression, linear SVM, or both. The trailing comma is required for a one-item tuple.

In [ ]:
FEATURE_EXTRACTORS_TO_RUN = ('tfidf', 'glove')
MODELS_TO_RUN = ('logistic_regression', 'linear_svm')

## Setup

Locate the project whether the notebook is opened from the repository root or from `notebooks/`, then import the reusable pipeline components.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = next((p for p in (Path.cwd(), Path.cwd().parent) if (p / 'modules').exists()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Run this notebook from the extracted project folder.')
sys.path.insert(0, str(PROJECT_ROOT))

In [ ]:
import subprocess
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-q', '-r',
    str(PROJECT_ROOT / 'requirements.txt'),
])

import importlib
import pandas as pd
import modules.plotter as plotter
plotter = importlib.reload(plotter)
from modules.data_downloader import load_dataset
from modules.data_preprocessor import prepare_data, save_preparation_audit
from modules.evaluator import evaluate_models, validate_models
from modules.feature_extractor import GloveFeatureExtractor, TfidfFeatureExtractor
from modules.models import LinearSVMModel, LogisticRegressionModel
from modules.text_eda import run_eda
from modules.trainer import (
    PreparedData, fit_final_models, save_feature_representations, tune_models
)

## 1. Exploratory data analysis

EDA uses the official training split. The test file is inspected only for schema and integrity checks.

In [ ]:
eda = run_eda(PROJECT_ROOT)
display(eda['quality'])
display(eda['class_distribution'])

## 2. Prepare shared data

Remove duplicates and train/test overlap, normalize the text, and create one stratified train-validation split shared by every configured model.

In [ ]:
train_raw, test_raw = load_dataset(PROJECT_ROOT / 'data')
train, validation, test, audit = prepare_data(train_raw, test_raw)
prepared_data = PreparedData(train, validation, test, audit)
save_preparation_audit(
    audit, PROJECT_ROOT / 'outputs/generated/preprocess/data_preparation_audit.json'
)
prepared_data.audit

## 3. Visualise extracted features before modelling

Fit visualization-only copies of the configured feature extractors on the clean training corpus, then display PCA/SVD, t-SNE, and UMAP projections. The same stratified 5,000-tweet sample is used for every plot. These reductions are diagnostic only: the classifiers below still receive the original TF-IDF and GloVe representations.

In [ ]:
# Reload local plotting code so rerunning this cell in an existing kernel
# cannot use a stale function signature from before the notebook update.
import importlib
import modules.plotter as plotter
plotter = importlib.reload(plotter)

visualization_extractors = {
    'tfidf': TfidfFeatureExtractor(),
    'glove': GloveFeatureExtractor(PROJECT_ROOT),
}
unknown_features = set(FEATURE_EXTRACTORS_TO_RUN) - visualization_extractors.keys()
if unknown_features:
    raise ValueError(f'Unknown feature selection: {sorted(unknown_features)}')
visualization_representations = {}
full_train = prepared_data.full_train
for feature_name in FEATURE_EXTRACTORS_TO_RUN:
    extractor = visualization_extractors[feature_name]
    extractor.fit(full_train['ProcessedTweet'])
    visualization_representations[feature_name] = (
        extractor.transform(full_train['ProcessedTweet']),
        extractor.label,
    )

feature_space_metadata = plotter.plot_feature_space_visualizations(
    PROJECT_ROOT,
    representations=visualization_representations,
    labels=full_train['Sentiment'],
)
display(feature_space_metadata)

In [ ]:
AVAILABLE_FEATURE_EXTRACTORS = {'tfidf', 'glove'}
MODEL_FACTORIES = {
    'logistic_regression': LogisticRegressionModel,
    'linear_svm': LinearSVMModel,
}
if not FEATURE_EXTRACTORS_TO_RUN or not MODELS_TO_RUN:
    raise ValueError('Select at least one feature extractor and one model')
unknown_features = set(FEATURE_EXTRACTORS_TO_RUN) - AVAILABLE_FEATURE_EXTRACTORS
if unknown_features:
    raise ValueError(f'Unknown feature selection: {sorted(unknown_features)}')
unknown_models = set(MODELS_TO_RUN) - MODEL_FACTORIES.keys()
if unknown_models:
    raise ValueError(f'Unknown model selection: {sorted(unknown_models)}')

def run_experiment(extractor):
    models = [MODEL_FACTORIES[name]() for name in MODELS_TO_RUN]
    if extractor.name == 'glove' and 'logistic_regression' in MODELS_TO_RUN:
        models[MODELS_TO_RUN.index('logistic_regression')] = LogisticRegressionModel((0.5, 1.0, 2.0))
    tuning = tune_models(extractor, models, prepared_data.train, PROJECT_ROOT)
    validation = validate_models(tuning, prepared_data.validation)
    training = fit_final_models(tuning, prepared_data.full_train, PROJECT_ROOT)
    save_feature_representations(
        training, prepared_data.full_train, prepared_data.test,
        validation.selected_model, PROJECT_ROOT
    )
    return evaluate_models(
        training, extractor, prepared_data.test, validation, PROJECT_ROOT
    )

## 4. Build, tune, and evaluate the models

### TF-IDF

Fit sparse word-level TF-IDF features and train only the classifiers selected above. Model pipelines use fresh extractor instances and the original high-dimensional representation, not the two-dimensional visualizations.

In [ ]:
results = {}
if 'tfidf' in FEATURE_EXTRACTORS_TO_RUN:
    results['tfidf'] = run_experiment(TfidfFeatureExtractor())

### GloVe-Twitter-100

Mean-pool pretrained Twitter embeddings and train the same classifier selection. The first run downloads the vectors into the project cache.

In [ ]:
if 'glove' in FEATURE_EXTRACTORS_TO_RUN:
    results['glove'] = run_experiment(GloveFeatureExtractor(PROJECT_ROOT))

## 5. Results

Display each feature-model metric table and confusion matrix, then create the consolidated four-combination comparison exactly once.

In [ ]:
for feature_name, result in results.items():
    plotter.plot_experiment_results(
        result, prepared_data.test['Sentiment'], PROJECT_ROOT
    )
    print(f"{result['feature_label']} selected: {result['selected_model']}")
    display(result['test_metrics'])

combined_test_metrics = pd.concat(
    [result['test_metrics'] for result in results.values()],
    ignore_index=True,
)
plotter.plot_model_comparison(
    combined_test_metrics,
    PROJECT_ROOT / 'outputs/graphics/comparison/feature_model_comparison.png',
)

## Generated artifacts

The preceding cells save all reproducibility artifacts under the project-level `features/` and `outputs/` folders.

In [ ]:
print('Feature matrices:', PROJECT_ROOT / 'features')
print('Figures and metrics:', PROJECT_ROOT / 'outputs')
print('Final report sources:', PROJECT_ROOT / 'report')

### Saved files

Feature representations, aligned labels, and metadata are saved in the project-level `features/` folder.  

Other generated artifacts are grouped under `outputs/`:
  
figures in `outputs/graphics/`, 

metrics and audits in `outputs/generated/`

fitted pipelines in `outputs/models/`.